# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR\^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by its Croissant schema, publicly available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading
Load the dataset Croissant schema and inspect high-level metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and create the Dataset object
dataset = mlc.Dataset(croissant_url)

# Print summary of the dataset using metadata attributes
print(f"Name        : {dataset.metadata.name}")
print(f"Description : {dataset.metadata.description}\n")
print(f"Citation    : {getattr(dataset.metadata, 'citeAs', None)}\n")
print(f"Published   : {getattr(dataset.metadata, 'datePublished', None)}\n")

## 2. Data Overview
Review available record sets and their fields using their Croissant `@id` for reference.

Here we enumerate record sets, show their `@id` and contained fields (columns) with `@id`, `name`, and `dataType`.

In [ ]:
# Find all available record set @id's in the schema
record_sets = dataset.metadata.record_sets
print(f"Total record sets: {len(record_sets)}")
for recset in record_sets:
    print(f"\nRecordSet @id: {recset.id}")
    print(f"  name      : {recset.name}")
    print(f"  Num fields: {len(recset.fields)}")
    print(f"  Fields:")
    for field in recset.fields:
        print(f"    - {field.id}")
        print(f"      name     : {field.name}")
        print(f"      dataType : {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load each record set as a Pandas DataFrame for downstream analysis. All entities are referenced by their `@id`.

In [ ]:
# List out all record sets
record_set_ids = [recset.id for recset in dataset.metadata.record_sets]
# Create a DataFrame for each record set
dataframes = {}

for recid in record_set_ids:
    records = list(dataset.records(record_set=recid))
    df = pd.DataFrame(records)
    dataframes[recid] = df
    print(f"Loaded DataFrame for {recid} (shape: {df.shape})")

# Show column names for the first record set
if len(record_set_ids):
    main_recset_id = record_set_ids[0]
    print(f"\nColumns in {main_recset_id}:")
    print(dataframes[main_recset_id].columns.tolist())
    dataframes[main_recset_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate some common EDA tasks using a numeric field in the main record set. All entity references use their Croissant `@id`.

- **Filter:** Select records where a numeric field (e.g., protein abundance or peptide count) exceeds a threshold.
- **Normalize:** Compute z-scores for the numeric field.
- **Group:** Group by a categorical field (e.g., protein description, accession number, or sample condition) if available for aggregation.

In [ ]:
# Select record set and some fields to analyze -- update the @id below to your main record set.
record_set_id = main_recset_id
df = dataframes[record_set_id]

# List all columns to help pick target fields (these are @id or schema field names)
print("Available columns:\n", df.columns.tolist())

# --- Example: pick first numeric field. Replace with actual @id in your exploration ---
# Here, we try to select 'coverage_percentage' or similar as a reasonable field for mass-spectrometry,
# but you should confirm the proper field name from the columns above.

# Try a few common field names that might be numeric; update as appropriate
possible_numeric_fields = [
    'coverage_percentage',
    'peptide_counts',
    'MW',
    'mw',
    'percent_coverage',
    'abundance',
    'normalized_abundance',
    # Add any field @id's you discover from the schema
]
numeric_field = None
for col in possible_numeric_fields:
    if col in df.columns:
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = df.select_dtypes(include='number').columns[0]  # Pick the first numeric column

print(f"Chosen numeric field (by id): {numeric_field}\n")

# Filter: e.g., above median
threshold = df[numeric_field].median()
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.3g}:")
print(filtered_df.head())

# Normalize field (z-score)
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, norm_col]].head())

# Optionally, group by a non-numeric field if available
# Here, we try 'description', 'accession', etc., or any other field present
possible_group_fields = [
    'description',
    'accession',
    'sample',
    'condition',
    # Add relevant field @id's
]
group_field = None
for col in possible_group_fields:
    if col in df.columns:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
We'll visualize the chosen numeric field using a histogram and plot its distribution after normalization. All axes and labels will reference field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")

plt.subplot(1,2,2)
sns.histplot(filtered_df[norm_col].dropna(), kde=True, color='orange')
plt.xlabel(f"{numeric_field} normalized (z-score)")
plt.title(f"Normalized {numeric_field}")

plt.tight_layout()
plt.show()

## 6. Conclusion
Using the `mlcroissant` library and the Croissant schema, we explored proteomics data, summarized available fields, loaded all record sets using their `@id`, filtered/normalized a numeric attribute, optionally grouped by a categorical one, and visualized distributions.

- The dataset schema (FAIR\^2) enables reproducible and standardized data loading and analysis.
- All data elements and attributes were accessed by their Croissant `@id` for clarity and traceability.
- This approach can be extended to more advanced analyses, including feature engineering and ML modeling.

For more information, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/).
